# DK Fødevarepris-Monitor
**Ugentlig opdatering af prisdatasæt fra nemlig.com**

Kør cellerne **én gang fra top til bund** – derefter er det nok at køre **celle 5** (Ugentlig opdatering) ved hvert besøg.

Data gemmes permanent på dit Google Drive under:
`Mit drev / food_price_monitor /`

## Trin 1 – Installer pakker
Skal kun køres én gang per Colab-session.

In [ ]:
!pip install -q requests beautifulsoup4 lxml
print('Pakker installeret.')

## Trin 2 – Montér Google Drive
Du bliver bedt om at godkende adgang første gang.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive monteret.')

## Trin 3 – Hent monitor-scriptet fra GitHub

In [ ]:
import os

REPO_DIR = '/content/fluffy-waffle'

if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/LasseLundqvist/fluffy-waffle.git {REPO_DIR}
    print('Repository hentet.')
else:
    !git -C {REPO_DIR} pull -q
    print('Repository opdateret.')

# Tilføj til Python-stien
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

## Trin 4 – Konfigurér datamapper på Drive
Data gemmes permanent under `Mit drev / food_price_monitor /`.

In [ ]:
from pathlib import Path
import dk_food_price_monitor as mon

# Peg alle datamapper mod Google Drive
DRIVE_ROOT = Path('/content/drive/MyDrive/food_price_monitor')

mon.DATA_DIR     = DRIVE_ROOT
mon.PRICES_DIR   = DRIVE_ROOT / 'daily_prices'
mon.INDEX_DIR    = DRIVE_ROOT / 'indices'
mon.EUROSTAT_DIR = DRIVE_ROOT / 'eurostat'

for folder in [mon.PRICES_DIR, mon.INDEX_DIR, mon.EUROSTAT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Opdatér log-fil-sti
import logging
mon.LOG_FILE = DRIVE_ROOT / 'monitor.log'
for handler in mon.log.handlers[:]:
    mon.log.removeHandler(handler)
mon.log.addHandler(logging.FileHandler(mon.LOG_FILE))
mon.log.addHandler(logging.StreamHandler(sys.stdout))

print(f'Datamapper klar: {DRIVE_ROOT}')

---
## Trin 5 – Ugentlig opdatering ▶
**Denne celle er alt du behøver at køre hver uge.**

Den henter aktuelle priser fra nemlig.com, beregner prisindekset og gemmer resultatet på dit Drive.

In [ ]:
from datetime import datetime
from dk_food_price_monitor import PriceMonitorPipeline

TODAY = datetime.now().strftime('%Y-%m-%d')
print(f'Starter ugentlig opdatering for {TODAY}\n')

pipe = PriceMonitorPipeline()

# 1. Scrape priser fra nemlig.com
observations = pipe.run_scrape(TODAY)

# 2. Beregn prisindeks
index = pipe.run_index(TODAY)

# 3. Vis rapport og eksportér CSV
pipe.run_report()

print(f'\nFærdig. Data gemt på Google Drive.')

## Trin 6 – Se historik og download CSV

In [ ]:
import pandas as pd

csv_path = DRIVE_ROOT / 'price_index_history.csv'

if csv_path.exists():
    df = pd.read_csv(csv_path, parse_dates=['date'])
    print(f'Datapunkter i alt: {len(df)}\n')
    display(df.tail(10))
else:
    print('Ingen historik endnu – kør trin 5 først.')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

csv_path = DRIVE_ROOT / 'price_index_history.csv'

if csv_path.exists() and len(df) >= 2:
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(df['date'], df['composite_index'], marker='o', linewidth=2, color='steelblue')
    ax.axhline(100, color='gray', linestyle='--', linewidth=0.8, label='Basis = 100')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%m-%Y'))
    plt.xticks(rotation=45, ha='right')
    ax.set_title('DK Fødevareprisindeks (nemlig.com)', fontsize=14)
    ax.set_ylabel('Indeks (base = 100)')
    ax.set_xlabel('Dato')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(DRIVE_ROOT / 'prisindeks_graf.png', dpi=150)
    plt.show()
    print('Graf gemt på Drive.')
else:
    print('Ikke nok datapunkter til graf endnu (kræver mindst 2 uger).')

In [ ]:
# Download CSV direkte til din computer
from google.colab import files

csv_path = DRIVE_ROOT / 'price_index_history.csv'

if csv_path.exists():
    files.download(str(csv_path))
else:
    print('Ingen CSV endnu – kør trin 5 først.')

---
## Appendiks – Eurostat HICP benchmark (valgfrit)
Hent officielle månedlige inflationstal fra Eurostat til sammenligning.
Ingen API-nøgle krævet.

In [ ]:
pipe.run_eurostat()
print('Eurostat HICP-data gemt på Drive.')

## Appendiks – Demo uden internet
Test hele pipeline med syntetiske priser – ingen netadgang nødvendig.

In [ ]:
pipe.run_demo()